# Semantic Search and Retrieval-Augmented Generation (RAG)

## Semantic Search with Language Models

### Dense Retrieval Example

In [1]:
import cohere
import numpy as np
import re
import pandas as pd
from tqdm import tqdm
from langchain.vectorstores import FAISS

In [2]:
import os
from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv(), override=True)

cohere_api_key = os.getenv("COHERE_API_KEY")
co = cohere.Client(cohere_api_key)

In [3]:
# Get text and chunk it

text = """
Interstellar is a 2014 epic science fiction drama film directed by Christopher Nolan, who co-wrote the screenplay with his brother Jonathan. It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine. Set in a dystopian future where Earth is suffering from catastrophic blight and famine, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.

The screenplay had its origins in a script Jonathan developed in 2007 and was originally set to be directed by Steven Spielberg. Theoretical physicist Kip Thorne was an executive producer and scientific consultant on the film, and wrote the tie-in book The Science of Interstellar. Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm. Filming began in late 2013 and took place in Alberta, Klaustur, and Los Angeles. Interstellar uses extensive practical and miniature effects, and the company DNEG created additional digital effects.

Interstellar premiered in Los Angeles on October 26, 2014. In the United States, it was first released on film stock, expanding to venues using digital projectors. The film received positive reviews from critics and grossed over $681 million worldwide ($705 million after subsequent re-releases), making it the tenth-highest-grossing film of 2014. Thorne's computer-generated depiction of a black hole in the film has also received commendation from astronomers and physicists.[4][5][6] Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades.
"""

# Split into a list of sentences
texts = text.split('.')

# Clean up to remove empty spaces and new lines
texts = [t.strip(' \n') for t in texts]

In [4]:
# Embed the text chunks

response = co.embed(texts=texts, input_type="search_document").embeddings
embeds = np.array(response)
print(embeds.shape)

(14, 4096)


In [5]:
# Build the search index

import faiss
dim = embeds.shape[1]
index = faiss.IndexFlatL2(dim)
print(index.is_trained)
index.add(np.float32(embeds))

True


In [6]:
# Search the index

def search(query, number_of_results=3):
  
  # 1. Get the query’s embedding
  query_embed = co.embed(texts=[query], 
                input_type="search_query",).embeddings[0]

  # 2. Retrieve the nearest neighbors
  distances, similar_item_ids = index.search(np.float32([query_embed]), number_of_results) 

  # 3. Format the results
  texts_np = np.array(texts) # Convert texts list to numpy for easier indexing
  results = pd.DataFrame(data={'texts': texts_np[similar_item_ids[0]], 
                              'distance': distances[0]})
  
  # 4. Print and return the results
  print(f"Query:'{query}'\nNearest neighbors:")
  return results


In [7]:
pd.set_option('display.max_colwidth', None)

query = "how precise was the science"
results = search(query)
results

Query:'how precise was the science'
Nearest neighbors:


,texts,distance
0,"Interstellar uses extensive practical and miniature effects, and the company DNEG created additional digital effects",11976.197266
1,Thorne's computer-generated depiction of a black hole in the film has also received commendation from astronomers and physicists,12197.380859
2,"Theoretical physicist Kip Thorne was an executive producer and scientific consultant on the film, and wrote the tie-in book The Science of Interstellar",12403.914062


#### Compare with simple lexical search method

In [8]:
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
import string

def bm25_tokenizer(text):
    tokenized_doc = []
    for token in text.lower().split():
        token = token.strip(string.punctuation)

        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
    return tokenized_doc

In [9]:
tokenized_corpus = []
for passage in tqdm(texts):
    tokenized_corpus.append(bm25_tokenizer(passage))

bm25 = BM25Okapi(tokenized_corpus)

100%|██████████| 14/14 [00:00<00:00, 46091.25it/s]


In [10]:
def keyword_search(query, top_k=3, num_candidates=14):
    print("Input question:", query)

    ##### BM25 search (lexical search) #####
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)
    
    print(f"Top-3 lexical search (BM25) hits")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))

In [11]:
keyword_search(query = "how precise was the science")

Input question: how precise was the science
Top-3 lexical search (BM25) hits
	1.497	Interstellar is a 2014 epic science fiction drama film directed by Christopher Nolan, who co-wrote the screenplay with his brother Jonathan
	1.497	Theoretical physicist Kip Thorne was an executive producer and scientific consultant on the film, and wrote the tie-in book The Science of Interstellar
	0.000	It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine


In [12]:
query = "how precise was the science"
results = co.rerank(query=query, documents=texts, top_n=3, return_documents=True)
results.results

[RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='The film received positive reviews from critics and grossed over $681 million worldwide ($705 million after subsequent re-releases), making it the tenth-highest-grossing film of 2014'), index=10, relevance_score=0.06669065),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='Theoretical physicist Kip Thorne was an executive producer and scientific consultant on the film, and wrote the tie-in book The Science of Interstellar'), index=4, relevance_score=0.026105916),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text="Thorne's computer-generated depiction of a black hole in the film has also received commendation from astronomers and physicists"), index=11, relevance_score=0.025908021)]

In [13]:
for idx, result in enumerate(results.results):
  print(idx, result.relevance_score , result.document.text)

0 0.06669065 The film received positive reviews from critics and grossed over $681 million worldwide ($705 million after subsequent re-releases), making it the tenth-highest-grossing film of 2014
1 0.026105916 Theoretical physicist Kip Thorne was an executive producer and scientific consultant on the film, and wrote the tie-in book The Science of Interstellar
2 0.025908021 Thorne's computer-generated depiction of a black hole in the film has also received commendation from astronomers and physicists


## Reranking

In [14]:
def keyword_and_reranking_search(query, top_k=3, num_candidates=10):
    print("Input question:", query)

    ##### BM25 search (lexical search) #####
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)
    
    print(f"Top-3 lexical search (BM25) hits")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))

   
    #Add re-ranking
    docs = [texts[hit['corpus_id']] for hit in bm25_hits]
    
    print(f"\nTop-3 hits by rank-API ({len(bm25_hits)} BM25 hits re-ranked)")
    results = co.rerank(query=query, documents=docs, top_n=top_k, return_documents=True)
    # print(results.results)
    for hit in results.results:
        # print(hit)
        print("\t{:.3f}\t{}".format(hit.relevance_score, hit.document.text.replace("\n", " ")))

In [15]:
keyword_and_reranking_search(query = "how precise was the science")

Input question: how precise was the science
Top-3 lexical search (BM25) hits
	1.497	Theoretical physicist Kip Thorne was an executive producer and scientific consultant on the film, and wrote the tie-in book The Science of Interstellar
	1.497	Interstellar is a 2014 epic science fiction drama film directed by Christopher Nolan, who co-wrote the screenplay with his brother Jonathan
	0.000	Filming began in late 2013 and took place in Alberta, Klaustur, and Los Angeles

Top-3 hits by rank-API (10 BM25 hits re-ranked)
	0.026	Theoretical physicist Kip Thorne was an executive producer and scientific consultant on the film, and wrote the tie-in book The Science of Interstellar
	0.004	Set in a dystopian future where Earth is suffering from catastrophic blight and famine, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind
	0.003	The screenplay had its origins in a script Jonathan developed in 2007 and was originally set to be dire

## Retrieval Augmented Generation

### Example: Grounded Generation with an LLM API

In [16]:
# Use embedding search to retrieve top documents
# Pass those to llm endpoint for grounded generation

query = "income generated"

# 1- Retrieval
# We’ll use embedding search. But ideally we’d do hybrid
results = search(query)

# 2- Grounded Generation
docs_dict = [{'text': text} for text in results['texts']]
response = co.chat(
    message = query,
    documents=docs_dict
)

print(response.text)

Query:'income generated'
Nearest neighbors:
The film grossed over $681 million worldwide ($705 million after subsequent re-releases).


In [17]:
print(response)

text='The film grossed over $681 million worldwide ($705 million after subsequent re-releases).' generation_id='909ae5a6-ae25-461c-b2b1-c70a7d798299' citations=[ChatCitation(start=9, end=88, text='grossed over $681 million worldwide ($705 million after subsequent re-releases)', document_ids=['doc_0'])] documents=[{'id': 'doc_0', 'text': 'The film received positive reviews from critics and grossed over $681 million worldwide ($705 million after subsequent re-releases), making it the tenth-highest-grossing film of 2014'}] is_search_required=None search_queries=None search_results=None finish_reason='COMPLETE' tool_calls=None chat_history=[Message_User(role='USER', message='income generated', tool_calls=None), Message_Chatbot(role='CHATBOT', message='The film grossed over $681 million worldwide ($705 million after subsequent re-releases).', tool_calls=None)] prompt=None meta=ApiMeta(api_version=ApiMetaApiVersion(version='1', is_deprecated=None, is_experimental=None), billed_units=ApiMetaB

### Example: RAG with Local Models

In [18]:
# Load the generation model 
from langchain import LlamaCpp

llm = LlamaCpp(
    model_path="../tmp/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

In [19]:
# Load the embedding model

from langchain.embeddings.huggingface import HuggingFaceEmbeddings

# Embedding model for converting text to numerical representations
embedding_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-small-en-v1.5'
)

/opt/conda/lib/python3.10/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in HuggingFaceInferenceAPIEmbeddings has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/var/tmp/ipykernel_50272/12974170.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embedding_model = HuggingFaceEmbeddings(
/opt/conda/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.au

In [20]:
# Set up a vector database

from langchain.vectorstores import FAISS

# Create a local vector database
db = FAISS.from_texts(texts, embedding_model)

In [21]:
# The RAG Prompt

from langchain import PromptTemplate

# Create a prompt template
template = """<|user|>
Relevant information:
{context}

Provide a concise answer for following question using the relevant information provided above:
{question}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

In [22]:
from langchain.chains import RetrievalQA

# RAG pipeline
rag = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=db.as_retriever(),
    chain_type_kwargs={
        "prompt": prompt
    },
    verbose=True
)

In [23]:
# Call the model and ask a question

# rag.invode("income generated") # failed with phi-3 and llama3.1
rag.invoke("How much income was generated by the movie?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



> Finished chain.


{'query': 'How much income was generated by the movie?',
 'result': 'The movie grossed over $681 million worldwide. After subsequent re-releases, it grossed over $705 million worldwide. |<|endoftext|>In this response, I will provide a concise answer for the question of how much income was generated by the movie.\n\nTherefore, The movie grossed over $681 million and after subsequent re-releases, it grossed over $705 million. |<|endoftext|>The final answer is: $681 million and $705 million.'}